# Practical 11: Music Genre Classification using an LSTM (GTZAN)

**Aim:** To classify music genres on GTZAN using an LSTM network (same MFCC features and song-level split as Practical 10) and compare its accuracy with the SimpleRNN.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Segment-level accuracy ~68-78%, song-level accuracy ~78-88%.; LSTM outperforms SimpleRNN by ~10-13% due to the gated cell state preserving rhythmic context over 130 time steps.

## Cell 1 - Install and import

**What it does:**
- `librosa` is the audio library used to load sound files and compute MFCC features.
- `StandardScaler` will normalise the MFCC values; `train_test_split` will split the songs.

*Expected output (illustrative):* TensorFlow version: 2.x.x

In [ ]:
!pip install -q librosa

import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

## Cell 2 - Settings and genre list

**What it does:**
- `DATA_PATH` is the GTZAN folder that contains 10 genre folders (blues, classical, ..., rock) each holding 100 `.wav` files (Kaggle: `genres_original`).
- Each 30-second track is cut into **10 segments of 3 seconds**. At 22,050 Hz a segment has 66,150 samples. 13 MFCCs are computed per frame.
- With `hop_length=512`, a 3-second segment gives 1 + 66150 // 512 = **130 frames**, so each segment becomes a (130, 13) matrix.

*Expected output (illustrative):* ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

In [ ]:
DATA_PATH = "genres_original"

SAMPLE_RATE = 22050
TRACK_DURATION = 30
SEGMENTS = 10
SEGMENT_SAMPLES = int(3 * SAMPLE_RATE)
N_MFCC = 13
N_FFT = 2048
HOP_LENGTH = 512

genres = sorted(d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d)))
print(genres)

## Cell 3 - Extract MFCC features (10 segments per song)

**What it does:**
- For every song: load it (30 s, 22,050 Hz), cut it into 3-second segments and compute MFCCs for each segment.
- `mfcc.T` transposes the result from (13, 130) to **(130 time steps, 13 features)** - the (batch, time_steps, features) shape that RNN/LSTM layers expect.
- `song_ids` remembers which song every segment came from. This is the key to a **leak-free split** and to song-level prediction later.
- The `try/except` skips corrupt files (the Kaggle GTZAN copy contains one broken jazz file).
- This cell takes a few minutes (about 1,000 songs).

*Expected output (illustrative):* X: (about 9990, 130, 13)  y: (about 9990,)
Number of songs: 999 or 1000

In [ ]:
features = []
labels = []
song_ids = []
song_counter = 0

for genre_index, genre in enumerate(genres):
    genre_dir = os.path.join(DATA_PATH, genre)
    for file_name in sorted(os.listdir(genre_dir)):
        if not file_name.endswith(".wav"):
            continue
        file_path = os.path.join(genre_dir, file_name)
        try:
            signal, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=TRACK_DURATION)
        except Exception as error:
            print("Skipping", file_name, "-", error)
            continue

        for seg in range(SEGMENTS):
            start = seg * SEGMENT_SAMPLES
            end = start + SEGMENT_SAMPLES
            if end > len(signal):
                break
            mfcc = librosa.feature.mfcc(
                y=signal[start:end], sr=sr, n_mfcc=N_MFCC,
                n_fft=N_FFT, hop_length=HOP_LENGTH)
            features.append(mfcc.T)
            labels.append(genre_index)
            song_ids.append(song_counter)
        song_counter += 1

X = np.array(features, dtype="float32")
y = np.array(labels)
song_ids = np.array(song_ids)

print("X:", X.shape, "y:", y.shape)
print("Number of songs:", song_counter)

## Cell 4 - Split by SONG (not by segment)

**What it does:**
- **Why split by song?** Segments of the same song sound alike. If random segments were split, a song's segments would appear in both train and test and accuracy would be falsely high (**data leakage**, from your PDF).
- Songs are divided 70% train / 10% validation / 20% test, stratified by genre; then the masks select all segments belonging to each group of songs.

*Expected output (illustrative):* Train segments: ~6,990  Val: ~1,000  Test: ~2,000 (about 70/10/20).

In [ ]:
unique_songs = np.unique(song_ids)
song_genre = np.array([y[song_ids == s][0] for s in unique_songs])

songs_train_val, songs_test, genre_train_val, _ = train_test_split(
    unique_songs, song_genre, test_size=0.2, random_state=42, stratify=song_genre)

songs_train, songs_val = train_test_split(
    songs_train_val, test_size=0.125, random_state=42, stratify=genre_train_val)

train_mask = np.isin(song_ids, songs_train)
val_mask = np.isin(song_ids, songs_val)
test_mask = np.isin(song_ids, songs_test)

x_train, y_train = X[train_mask], y[train_mask]
x_val, y_val = X[val_mask], y[val_mask]
x_test, y_test = X[test_mask], y[test_mask]
test_song_ids = song_ids[test_mask]

print("Train segments:", len(x_train), " Val:", len(x_val), " Test:", len(x_test))

## Cell 5 - Normalise MFCCs with StandardScaler

**What it does:**
- MFCC coefficients have very different ranges. StandardScaler rescales each coefficient to mean 0 and standard deviation 1.
- The scaler is **fitted only on the training data** and then applied to validation and test data, so no information from the test set leaks in.

*Expected output (illustrative):* (about 6990, 130, 13) 0.0 1.0

In [ ]:
scaler = StandardScaler()
scaler.fit(x_train.reshape(-1, N_MFCC))

def normalise(data):
    flat = scaler.transform(data.reshape(-1, N_MFCC))
    return flat.reshape(data.shape).astype("float32")

x_train = normalise(x_train)
x_val = normalise(x_val)
x_test = normalise(x_test)

print(x_train.shape, round(float(x_train.mean()), 3), round(float(x_train.std()), 3))

## Cell 6 - Build the LSTM model

**What it does:**
- The first `LSTM(128, return_sequences=True)` outputs a hidden state for **every** time step (130 x 128) so that the second LSTM receives a sequence. The second `LSTM(64)` returns only its final state (64 numbers). Its gated cell state preserves rhythmic context over the 130 steps.
- `Dropout(0.3)` randomly disables 30% of the units during training to reduce overfitting.
- `Dense(64, relu)` combines the features; `Dense(10, softmax)` outputs the probability of each of the 10 genres.
- Layer sizes (128 -> 64 -> Dense 64 -> Dense 10) are exactly those in your PDF; the dropout rate 0.3 is an addition.

*Expected output (illustrative):* Output shapes (130,128) -> (130,128) -> (64) -> (64) -> (64) -> (10). Total params about 126,922 (LSTM 128: 72,704; LSTM 64: 49,408; Dense 64: 4,160; Dense 10: 650).

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(130, N_MFCC)),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])

model.summary()

## Cell 7 - Compile and train

**What it does:**
- `clipnorm=1.0` (gradient clipping) prevents exploding gradients, which are common in recurrent networks.
- The validation data are whole songs that were kept out of training. `EarlyStopping(patience=10)` stops when validation loss stops improving and restores the best weights.
- LSTM is about 2x slower per epoch than SimpleRNN (from your PDF) but significantly more accurate.

*Expected output (illustrative):* Logs per epoch; validation accuracy reaches roughly 0.68-0.78 (segment level, from your PDF).

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True)

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
)

## Cell 8 - Accuracy and loss graphs

**What it does:**
- A growing gap between train and validation curves signals overfitting (GTZAN is small - only 700 training songs).

*Expected output (illustrative):* Two curves; training accuracy rises above validation accuracy.

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 9 - Segment-level evaluation

**What it does:**
- Segment-level accuracy = how often each individual 3-second clip is classified correctly.
- The confusion matrix and report show which genres are mixed up (rock<->metal, country<->blues, from your PDF).

*Expected output (illustrative):* Segment-level accuracy ~0.68-0.78 (from your PDF).

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("Segment-level test accuracy:", round(test_acc, 4))

seg_probs = model.predict(x_test, verbose=0)
seg_pred = np.argmax(seg_probs, axis=1)

cm = confusion_matrix(y_test, seg_pred)
ConfusionMatrixDisplay(cm, display_labels=genres).plot(cmap="Blues", xticks_rotation=45)
plt.title("Segment-level Confusion Matrix")
plt.show()

print(classification_report(y_test, seg_pred, target_names=genres, digits=3))

## Cell 10 - Song-level prediction (average the 10 segments)

**What it does:**
- A real user wants the genre of a whole **song**. For each test song the predicted probabilities of its 10 segments are **averaged** and the genre with the highest average probability wins.
- Averaging cancels out random segment errors, so song-level accuracy is higher than segment-level accuracy.

*Expected output (illustrative):* Song-level accuracy ~0.78-0.88 (from your PDF).

In [ ]:
song_true, song_pred = [], []

for s in np.unique(test_song_ids):
    mask = test_song_ids == s
    average_probs = seg_probs[mask].mean(axis=0)
    song_pred.append(int(np.argmax(average_probs)))
    song_true.append(int(y_test[mask][0]))

song_true = np.array(song_true)
song_pred = np.array(song_pred)

print("Song-level test accuracy:", round(float(np.mean(song_true == song_pred)), 4))

cm = confusion_matrix(song_true, song_pred, labels=list(range(10)))
ConfusionMatrixDisplay(cm, display_labels=genres).plot(cmap="Blues", xticks_rotation=45)
plt.title("Song-level Confusion Matrix")
plt.show()

print(classification_report(song_true, song_pred, labels=list(range(10)),
                            target_names=genres, digits=3, zero_division=0))

## Cell 11 - Save the model

**What it does:**
- Stores the model; remember to keep the fitted `scaler` too if you want to classify new songs later.

*Expected output (illustrative):* Model saved.

In [ ]:
model.save("gtzan_lstm.keras")
print("Model saved.")

## Result / Conclusion

The LSTM on MFCC features reached about 68-78% segment-level and 78-88% song-level accuracy, clearly better than the SimpleRNN, thanks to its ability to retain rhythmic context across time steps.